# 03 — Resumable tracked ViTPose extraction (safe pilot first)

This notebook starts one fresh worker process per source video. Each worker loads YOLO + ByteTrack + your existing ViTPose checkpoint, processes one clip, writes an atomic NPZ, and exits. Worker crashes/timeouts are recorded in logs and the per-video CSV; Jupyter does not run the heavy inference loop.

**Defaults:** five pilot classes, one video per class, CPU inference, up to 120 decoded frames per video, ViTPose every 4 frames, at most two people posed per sampled frame. This is only a pilot.

**After reviewing the pilot and approving a full run:** set `RUN_FULL_MANIFEST=True` **and** `MAX_DECODED_FRAMES=0`. The worker records whether it actually reached video EOF. Notebook 04 will refuse to call the dataset complete if any source clip was capped or failed. Do not change the original split.

In [ ]:
from pathlib import Path
from datetime import datetime
import json, os, sys, subprocess, hashlib, re, time
import pandas as pd

PROJECT_DIR=Path(r"C:\Users\admin\ViTPose_Action_Recognition")
NOTEBOOK2_DIR=PROJECT_DIR/"Notebook_2_Improved_v2"
SPLIT_PATH=PROJECT_DIR/"data"/"processed"/"sequences_group_split_v1"/"video_split_manifest.csv"
VITPOSE_CHECKPOINT=PROJECT_DIR/"checkpoints"/"epoch_209.pth"
YOLO_CANDIDATES=[PROJECT_DIR/"notebooks"/"yolo11n.pt",PROJECT_DIR/"yolo11n.pt"]
YOLO_WEIGHTS=next((p for p in YOLO_CANDIDATES if p.is_file()),None)
OUT_DIR=NOTEBOOK2_DIR/"data"/"tracked_keypoints_v2"
REPORT_DIR=NOTEBOOK2_DIR/"outputs"/"reports"/"tracked_pose_extraction_v2"
LOG_DIR=NOTEBOOK2_DIR/"logs"/"tracked_pose_extraction_v2"
SRC_DIR=NOTEBOOK2_DIR/"src"
for p in [OUT_DIR,REPORT_DIR,LOG_DIR,SRC_DIR]: p.mkdir(parents=True,exist_ok=True)
assert SPLIT_PATH.is_file(),f"Existing group split manifest missing: {SPLIT_PATH}"
assert VITPOSE_CHECKPOINT.is_file(),f"ViTPose checkpoint missing: {VITPOSE_CHECKPOINT}"
assert YOLO_WEIGHTS is not None,"YOLO weights not found in expected Notebook 1 paths"

# SAFETY DEFAULTS: only a small pilot. Switch to full manifest only after manual review.
RUN_FULL_MANIFEST=False
PILOT_CLASSES=["Typing","Knitting","BoxingPunchingBag","Biking","Rowing"]
VIDEOS_PER_CLASS=1
MAX_DECODED_FRAMES=120  # set 0 only for full-length videos after pilot approval
FRAME_STEP=4
MAX_PEOPLE_PER_FRAME=2
CONF=0.20
IMGSZ=320
WORKER_DEVICE="cpu"  # change to cuda:0 only after CPU pilot works and GPU is available
WORKER_TIMEOUT_SECONDS=1800

split_df=pd.read_csv(SPLIT_PATH)
required={"video_path","label","split","video_id","recording_group_key"}
missing=required-set(split_df.columns)
assert not missing,f"Split manifest missing columns: {sorted(missing)}"
if RUN_FULL_MANIFEST:
    selected=split_df.copy()
else:
    selected_rows=[]
    for label in PILOT_CLASSES:
        rows=split_df[split_df.label.astype(str).str.casefold().eq(label.casefold())].sort_values("video_path")
        if not rows.empty: selected_rows.extend(rows.head(VIDEOS_PER_CLASS).to_dict("records"))
    selected=pd.DataFrame(selected_rows).drop_duplicates("video_path")
assert not selected.empty,"No videos selected; check PILOT_CLASSES against manifest"

WORKER=SRC_DIR/"tracked_vitpose_worker_v2.py"
WORKER.write_text('\nimport os, sys\nfrom pathlib import Path\n# Prefer PyTorch\'s bundled OpenMP DLL in the child process without modifying\n# the parent Jupyter environment or deleting any DLL/package.\ntorch_lib = Path(sys.prefix) / "Lib" / "site-packages" / "torch" / "lib"\nimport ctypes\n_openmp_handle = None\nif os.name == "nt":\n    _torch_iomp = torch_lib / "libiomp5md.dll"\n    if _torch_iomp.is_file():\n        try:\n            _openmp_handle = ctypes.WinDLL(str(_torch_iomp))\n            print("OPENMP_PRELOADED_FROM", _torch_iomp, flush=True)\n        except OSError as _dll_error:\n            print("OPENMP_PRELOAD_WARNING", repr(_dll_error), flush=True)\n_dll_handles = []\nif torch_lib.is_dir():\n    os.environ["PATH"] = str(torch_lib) + os.pathsep + os.environ.get("PATH", "")\n    if hasattr(os, "add_dll_directory"):\n        try:\n            _dll_handles.append(os.add_dll_directory(str(torch_lib)))\n        except OSError:\n            pass\nfor _name in ["OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS", "NUMEXPR_NUM_THREADS"]:\n    os.environ[_name] = "1"\nos.environ["PYTHONUNBUFFERED"] = "1"\n\nimport json, time, gc, traceback, hashlib, re\nfrom pathlib import Path\n\ncfg=json.loads(Path(sys.argv[1]).read_text(encoding="utf-8"))\ncap=None\ntry:\n    import torch\n    torch.set_num_threads(1)\n    import cv2\n    cv2.setNumThreads(1)\n    import numpy as np\n    from ultralytics import YOLO\n    project=Path(cfg["project_dir"])\n    sys.path.insert(0,str(project))\n    from src.vitpose_model import ViTPose, get_affine_transform, decode_heatmaps, IMG_H, IMG_W, IMG_MEAN, IMG_STD\n    device=torch.device(cfg.get("device","cpu"))\n    print("WORKER_START",flush=True)\n    print("device",device,"weights",cfg["yolo_weights"],flush=True)\n    # Load models once for this video; the process exits when the video finishes.\n    yolo=YOLO(cfg["yolo_weights"])\n    pose=ViTPose().to(device)\n    try: ckpt=torch.load(cfg["vitpose_checkpoint"],map_location=device,weights_only=True)\n    except TypeError: ckpt=torch.load(cfg["vitpose_checkpoint"],map_location=device)\n    if "model_state" not in ckpt: raise KeyError("ViTPose checkpoint does not contain model_state")\n    pose.load_state_dict(ckpt["model_state"],strict=True); pose.eval()\n    print("MODELS_LOADED",flush=True)\n\n    def estimate_pose(rgb,box):\n        x1,y1,x2,y2=map(float,box); h,w=rgb.shape[:2]\n        x1=float(np.clip(x1,0,w-1)); x2=float(np.clip(x2,0,w))\n        y1=float(np.clip(y1,0,h-1)); y2=float(np.clip(y2,0,h))\n        bw,bh=x2-x1,y2-y1\n        if bw<2 or bh<2: return None\n        center=np.array([(x1+x2)/2,(y1+y2)/2],dtype=np.float32)\n        scale=max(bw,bh*IMG_W/IMG_H)*1.25\n        trans=get_affine_transform(center,scale,0.0,(IMG_W,IMG_H))\n        crop=cv2.warpAffine(rgb,trans,(IMG_W,IMG_H)).astype(np.float32)/255.0\n        crop=(crop-IMG_MEAN)/IMG_STD\n        tensor=torch.from_numpy(crop.transpose(2,0,1).copy()).float().unsqueeze(0).to(device)\n        with torch.inference_mode(): hm=pose(tensor)[0].float().cpu().numpy()\n        kp=decode_heatmaps(hm,center,scale).astype(np.float32)\n        del tensor,hm,crop\n        return kp if kp.shape==(17,3) else None\n\n    video=Path(cfg["video_path"])\n    cap=cv2.VideoCapture(str(video))\n    if not cap.isOpened(): raise RuntimeError("Could not open video")\n    total=int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)\n    fps=float(cap.get(cv2.CAP_PROP_FPS) or 0.0)\n    sample_frames=[]; sample_observations=[]; track_counts={}; frame_i=0; started=time.time(); reached_eof=False\n    while True:\n        if int(cfg.get("max_decoded_frames",0))>0 and frame_i>=int(cfg["max_decoded_frames"]): break\n        ok,frame=cap.read()\n        if not ok:\n            reached_eof=True\n            break\n        # Update the tracker on each decoded frame for temporal association.\n        result=yolo.track(frame,persist=True,tracker="bytetrack.yaml",classes=[0],conf=float(cfg["conf"]),\n                          imgsz=int(cfg["imgsz"]),device=cfg.get("device","cpu"),max_det=int(cfg["max_det"]),verbose=False)[0]\n        if frame_i % int(cfg["frame_step"]) == 0:\n            observations=[]\n            if result.boxes is not None and len(result.boxes)>0:\n                boxes=result.boxes.xyxy.cpu().numpy(); confs=result.boxes.conf.cpu().numpy()\n                ids=result.boxes.id.cpu().numpy().astype(np.int64) if result.boxes.id is not None else np.full(len(boxes),-1,dtype=np.int64)\n                areas=np.maximum(0,boxes[:,2]-boxes[:,0])*np.maximum(0,boxes[:,3]-boxes[:,1])\n                keep=np.argsort(areas)[::-1][:int(cfg["max_people_per_frame"])]\n                rgb=cv2.cvtColor(frame,cv2.COLOR_BGR2RGB)\n                for ix in keep:\n                    tid=int(ids[ix])\n                    kp=estimate_pose(rgb,boxes[ix])\n                    if kp is None: continue\n                    observations.append({"track_id":tid,"keypoints":kp,"box":boxes[ix].astype(np.float32),\n                                         "score":float(confs[ix]),"area":float(areas[ix])})\n                    if tid>=0: track_counts[tid]=track_counts.get(tid,0)+1\n                del rgb\n            sample_frames.append(frame_i); sample_observations.append(observations)\n        frame_i+=1\n        del result\n        if frame_i%50==0: print(f"PROGRESS frames={frame_i} samples={len(sample_frames)} tracks={len(track_counts)}",flush=True)\n        if frame_i%100==0: gc.collect()\n    cap.release(); cap=None\n\n    target_id=max(track_counts,key=track_counts.get) if track_counts else -1\n    n=len(sample_frames)\n    keypoints=np.full((n,17,3),np.nan,dtype=np.float32)\n    boxes=np.full((n,4),np.nan,dtype=np.float32)\n    has_person=np.zeros(n,dtype=bool)\n    target_ids=np.full(n,-1,dtype=np.int64)\n    det_scores=np.zeros(n,dtype=np.float32)\n    all_kps=[]; all_boxes=[]; all_frames=[]; all_ids=[]; all_scores=[]\n    for j,obs_list in enumerate(sample_observations):\n        for obs in obs_list:\n            all_kps.append(obs["keypoints"]); all_boxes.append(obs["box"])\n            all_frames.append(sample_frames[j]); all_ids.append(obs["track_id"]); all_scores.append(obs["score"])\n        match=next((o for o in obs_list if o["track_id"]==target_id and target_id>=0),None)\n        if match is not None:\n            keypoints[j]=match["keypoints"]; boxes[j]=match["box"]\n            has_person[j]=bool(np.isfinite(match["keypoints"][:,:2]).all(axis=1).sum()>=8)\n            target_ids[j]=target_id; det_scores[j]=match["score"]\n    all_kps=np.asarray(all_kps,dtype=np.float32).reshape(-1,17,3)\n    all_boxes=np.asarray(all_boxes,dtype=np.float32).reshape(-1,4)\n    all_frames=np.asarray(all_frames,dtype=np.int32); all_ids=np.asarray(all_ids,dtype=np.int64); all_scores=np.asarray(all_scores,dtype=np.float32)\n    out=Path(cfg["output_path"]); out.parent.mkdir(parents=True,exist_ok=True)\n    temp=out.with_name(out.stem+f".{os.getpid()}.tmp.npz")\n    np.savez_compressed(temp,keypoints=keypoints,boxes=boxes,frame_indices=np.asarray(sample_frames,dtype=np.int32),\n        has_person=has_person,track_ids=target_ids,detection_scores=det_scores,\n        all_keypoints=all_kps,all_boxes=all_boxes,all_frame_indices=all_frames,all_track_ids=all_ids,all_detection_scores=all_scores,\n        label=np.asarray(cfg["label"]),video_path=np.asarray(cfg["video_path"]),target_track_id=np.asarray(target_id),\n        fps=np.asarray(fps,dtype=np.float32),total_frames=np.asarray(total,dtype=np.int32),frame_step=np.asarray(cfg["frame_step"],dtype=np.int32),\n        person_selection=np.asarray("most_pose_observations_by_persistent_track_id"),\n        complete_video=np.asarray(reached_eof,dtype=np.bool_),max_decoded_frames=np.asarray(cfg.get("max_decoded_frames",0),dtype=np.int32),\n        decoded_frames=np.asarray(frame_i,dtype=np.int32),max_people_per_frame=np.asarray(cfg["max_people_per_frame"],dtype=np.int32))\n    os.replace(temp,out)\n    result_obj={"status":"success","label":cfg["label"],"video_path":cfg["video_path"],"output_path":str(out),\n        "decoded_frames":frame_i,"sampled_frames":n,"pose_successes":int(has_person.sum()),\n        "pose_success_rate":float(has_person.mean()) if n else 0.0,"tracked_ids":len(track_counts),\n        "target_track_id":int(target_id),"all_person_observations":len(all_ids),"complete_video":bool(reached_eof),\n        "seconds":round(time.time()-started,2)}\n    Path(cfg["result_path"]).write_text(json.dumps(result_obj,indent=2),encoding="utf-8")\n    print("WORKER_SUCCESS",json.dumps(result_obj),flush=True)\nexcept BaseException as exc:\n    print("WORKER_EXCEPTION",repr(exc),flush=True); traceback.print_exc(); sys.exit(2)\nfinally:\n    if cap is not None: cap.release()\n    gc.collect()\n',encoding="utf-8")
print("Mode:","FULL MANIFEST" if RUN_FULL_MANIFEST else "PILOT")
print("Videos selected:",len(selected))
print("Device:",WORKER_DEVICE,"| max decoded frames:",MAX_DECODED_FRAMES,"| pose frame step:",FRAME_STEP)
print("Outputs:",OUT_DIR)

In [ ]:
import numpy as np

def normalized_video_key(value):
    return str(Path(str(value)).resolve()).replace("/","\\").casefold()

def tracked_output_path(video_path,label):
    safe=re.sub(r"[^A-Za-z0-9_-]+","_",str(label)).strip("_") or "unknown"
    digest=hashlib.sha1(normalized_video_key(video_path).encode("utf-8")).hexdigest()[:12]
    return OUT_DIR/f"{safe}__{Path(str(video_path)).stem}__{digest}.npz"

def existing_is_complete(path,video_path):
    if not path.is_file(): return False
    try:
        with np.load(path,allow_pickle=False) as d:
            need={"keypoints","boxes","frame_indices","has_person","track_ids","all_keypoints","all_track_ids","label","video_path",
                  "complete_video","max_decoded_frames","frame_step","max_people_per_frame","decoded_frames"}
            if not need.issubset(d.files): return False
            n=len(d["frame_indices"])
            path_matches=normalized_video_key(d["video_path"].item())==normalized_video_key(video_path)
            params_match=(int(d["frame_step"])==FRAME_STEP and int(d["max_people_per_frame"])==MAX_PEOPLE_PER_FRAME)
            if MAX_DECODED_FRAMES==0:
                coverage_ok=bool(d["complete_video"].item())
            else:
                stored_limit=int(d["max_decoded_frames"])
                coverage_ok=bool(d["complete_video"].item()) or (stored_limit==MAX_DECODED_FRAMES and int(d["decoded_frames"])>=MAX_DECODED_FRAMES)
            return (d["keypoints"].shape==(n,17,3) and d["boxes"].shape==(n,4)
                    and d["has_person"].shape==(n,) and d["track_ids"].shape==(n,)
                    and path_matches and params_match and coverage_ok)
    except Exception:
        return False

def run_one_video(rec,position,total):
    video_path=Path(str(rec["video_path"])); label=str(rec["label"])
    out=tracked_output_path(video_path,label)
    if existing_is_complete(out,video_path):
            return {"status":"skipped_existing","label":label,"video_path":str(video_path),"output_path":str(out)}
    if not video_path.is_file():
        return {"status":"missing_video","label":label,"video_path":str(video_path),"output_path":""}
    rid=hashlib.sha1(str(video_path).encode("utf-8")).hexdigest()[:10]
    cfg_path=LOG_DIR/f"config_{rid}.json"; result_path=LOG_DIR/f"result_{rid}.json"
    stdout_path=LOG_DIR/f"stdout_{rid}.log"; stderr_path=LOG_DIR/f"stderr_{rid}.log"
    result_path.unlink(missing_ok=True)
    cfg={"project_dir":str(PROJECT_DIR),"video_path":str(video_path),"label":label,
         "yolo_weights":str(YOLO_WEIGHTS),"vitpose_checkpoint":str(VITPOSE_CHECKPOINT),
         "output_path":str(out),"result_path":str(result_path),"frame_step":FRAME_STEP,
         "max_decoded_frames":MAX_DECODED_FRAMES,"max_people_per_frame":MAX_PEOPLE_PER_FRAME,
         "conf":CONF,"imgsz":IMGSZ,"max_det":10,"device":WORKER_DEVICE}
    cfg_path.write_text(json.dumps(cfg,indent=2),encoding="utf-8")
    env=os.environ.copy()
    for key in ["OMP_NUM_THREADS","MKL_NUM_THREADS","OPENBLAS_NUM_THREADS","NUMEXPR_NUM_THREADS"]: env[key]="1"
    torch_lib=Path(sys.prefix)/"Lib"/"site-packages"/"torch"/"lib"
    if torch_lib.is_dir(): env["PATH"]=str(torch_lib)+os.pathsep+env.get("PATH","")
    try:
        proc=subprocess.run([sys.executable,str(WORKER),str(cfg_path)],cwd=str(PROJECT_DIR),env=env,
            capture_output=True,text=True,errors="replace",timeout=WORKER_TIMEOUT_SECONDS)
        stdout=proc.stdout or ""; stderr=proc.stderr or ""; rc=proc.returncode
    except subprocess.TimeoutExpired as exc:
        stdout=exc.stdout or ""; stderr=(exc.stderr or "")+"\nTIMEOUT"; rc=124
    stdout_path.write_text(stdout,encoding="utf-8",errors="replace")
    stderr_path.write_text(stderr,encoding="utf-8",errors="replace")
    if rc==0 and result_path.is_file():
        result=json.loads(result_path.read_text(encoding="utf-8"))
        result["worker_return_code"]=rc
        return result
    print("  WORKER FAILED; return code:",rc,flush=True)
    print("  stderr log:",stderr_path,flush=True)
    return {"status":"worker_failed","label":label,"video_path":str(video_path),"output_path":"",
            "worker_return_code":rc,"stdout_log":str(stdout_path),"stderr_log":str(stderr_path),"stderr_tail":stderr[-2000:]}

report_path=REPORT_DIR/"tracked_pose_extraction_report.csv"
if report_path.is_file():
    prior=pd.read_csv(report_path).to_dict("records")
else:
    prior=[]
results={str(r.get("video_path")):r for r in prior}
for pos,rec in enumerate(selected.to_dict("records"),1):
    row=run_one_video(rec,pos,len(selected))
    results[str(row["video_path"])]=row
    pd.DataFrame(list(results.values())).to_csv(report_path,index=False)
    if pos % 25 == 0 or pos == len(selected):
        print(f"Progress: {pos}/{len(selected)} videos recorded; latest status={row['status']}", flush=True)
print("\nEXTRACTION PILOT FINISHED")
print(pd.DataFrame(list(results.values())).tail(10).to_string(index=False))
print("Report:",report_path)

## Review before full extraction

Review at least several saved NPZs and tracking logs. Confirm the selected `target_track_id` is the intended person, the track is reasonably persistent, and the skeleton overlays are anatomically plausible. The output preserves all observed person keypoints in `all_keypoints` / `all_track_ids` as well as the selected target sequence in `keypoints` / `track_ids`.

For a full run only after approval: set `RUN_FULL_MANIFEST=True` and `MAX_DECODED_FRAMES=0`. The loop checkpoints the CSV after every video and skips valid completed NPZs. It can still take many hours. A worker crash is logged per video and does not require rerunning successful outputs.